### Middleware
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import HumanMessage,SystemMessage
from langgraph.checkpoint.memory import InMemorySaver

# Based on Message count
agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger={"messages":10},
            keep=("messages",4)
        )
    ]

)

In [5]:
# Provide runtime configuration
config = {"configurable":{"thread_id":"test_1"}}


In [8]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {response['messages'][-1]}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nAnswer the user's arithmetic questions.\n\n## SUMMARY\n- User asked: “What is 2+2?” → AI answered: 4.\n- User asked: “What is 10*5?” → AI answered: 50.\n- User asked: “What is 100/4?” → AI answered: 25.\n- User asked: “What is 15-7?” → No answer yet.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nProvide the answer to “What is 15-7?” (the answer is 8).", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='e8ef662b-ea70-4b25-b611-7e54cdb6ea8a'), AIMessage(content='15\u202f−\u202f7\u202f=\u202f8.', additional_kwargs={'reasoning_content': 'User asks simple arithmetic. Answer 8.'}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 160, 'total_tokens': 189, 'completion_time': 0.061151846, 'completion_tokens_details': {'reasoning_tokens': 10}, 'prompt_time': 0.009185042, 'prompt_tokens_details': None, 'queue_time': 0.25053712, 'total_t

#### Summarize based on Token Count

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import HumanMessage,SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

# Based on Message count
agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("tokens", 500),
            keep=("tokens",200)
        )
    ],
    tools=[search_hotels]

)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [ ]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages") # Here token count is appproximate
    print(f"{(response['messages'])}")

Paris: ~283 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='5edc135d-82b4-48ac-861f-458779d9514d'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants hotels in Paris. Use the function search_hotels with city "Paris".', 'tool_calls': [{'id': 'fc_9642e6aa-2e5b-43af-8126-25f766ae94c9', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 129, 'total_tokens': 176, 'completion_time': 0.098283799, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.005052151, 'prompt_tokens_details': None, 'queue_time': 0.286376259, 'total_time': 0.10333595}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c800245357', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1de16-d2c0-7532-b4d8-a1abdbc8

#### Fraction
`fraction` represents a percentage (proportion) of the model's context window, not a fixed number of tokens.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = "groq:openai/gpt-oss-120b",
            trigger= ("fraction", 0.005), # 0.5% of total context size
            keep= ("fraction", 0.002)
        ),
    ],
    tools=[search_hotels]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # context size
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~215 tokens (0.1680%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='a65d2ee4-1707-4b51-a448-f7bb639983d1'), AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants hotels in Paris. Use function search_hotels with city "Paris".', 'tool_calls': [{'id': 'fc_46012bc6-5c4c-4710-a13f-6367de9d7282', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 120, 'total_tokens': 165, 'completion_time': 0.095321268, 'completion_tokens_details': {'reasoning_tokens': 17}, 'prompt_time': 0.005194993, 'prompt_tokens_details': None, 'queue_time': 0.318463233, 'total_time': 0.100516261}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4200b3f836', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1df6b-46a8-7434-b130-946c092d8740-0

### Human In the Loop MiddleWare
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:
- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [34]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

@tool
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"
@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[send_email_tool,read_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on ={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            },
        )
    ]
)


In [ ]:
config = {"configurable": {"thread_id": "test-1111"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='81a20e2a-6be3-4c46-a047-7eeb434f90a5'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using send_email_tool. Provide parameters.', 'tool_calls': [{'id': 'fc_966036da-b29f-4aaf-8102-523338ab5be0', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 174, 'total_tokens': 234, 'completion_time': 0.125917175, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.054368468, 'prompt_tokens_details': None, 'queue_time': 0.371081647, 'total_time': 0.180285643}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_19b184c447', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': No

In [43]:
# Approve the request
from langgraph.types import Command
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: Your email has been sent successfully. Let me know if there’s anything else I can help with!


In [47]:
# Edit 
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: I attempted to send the email you requested. The tool call was overridden by a human reviewer, and an email was instead sent to **correct@email.com** with the subject **“Corrected Subject”** and the body **“This was edited by human before sending.”**


In [49]:
# Reject 
from langgraph.types import Command
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"reject"}
                ]
            }
        ),
        config=config
    )
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: I tried to send the email, but the request to use the email‑sending tool was rejected. Would you like me to attempt sending it again, or is there something else I can help you with?
